### 数据导入后查看数据

In [150]:
import pandas as pd

# 读取 events.csv
events = pd.read_csv('events.csv')

# 1. 查看数据基本信息
print("数据形状:", events.shape)
print("\n列名:\n", events.columns.tolist())
print("\n前5行:\n", events.head())

# 2. 检查 event_type 的取值和数量
print("\n事件类型分布:\n", events['event_type'].value_counts())

# 3. 检查关键列是否有缺失
print("\n缺失值统计:\n", events[['session_id', 'event_type', 'timestamp']].isnull().sum())

# 4. 查看 timestamp 的格式示例
print("\n时间戳示例:", events['timestamp'].iloc[0])

数据形状: (760958, 10)

列名:
 ['event_id', 'session_id', 'timestamp', 'event_type', 'product_id', 'qty', 'cart_size', 'payment', 'discount_pct', 'amount_usd']

前5行:
    event_id  session_id            timestamp   event_type  product_id  qty  \
0         1           1  2021-12-27T00:08:36    page_view        93.0  NaN   
1         2           1  2021-12-27T00:16:36    page_view      1005.0  NaN   
2         3           1  2021-12-27T00:18:01  add_to_cart      1005.0  1.0   
3         4           1  2021-12-27T00:45:36    page_view       918.0  NaN   
4         5           1  2021-12-27T01:03:36    page_view       946.0  NaN   

   cart_size payment  discount_pct  amount_usd  
0        NaN     NaN           NaN         NaN  
1        NaN     NaN           NaN         NaN  
2        NaN     NaN           NaN         NaN  
3        NaN     NaN           NaN         NaN  
4        NaN     NaN           NaN         NaN  

事件类型分布:
 event_type
page_view      539343
add_to_cart    143126
checkout   

In [151]:
print(events_clean['timestamp'].min(), events_clean['timestamp'].max())

2020-01-01 00:32:18 2025-11-01 01:50:04


### 数据清洗

In [152]:
# 检查每一列的缺失值数量
print(events.isnull().sum())

# 删除 event_id 或 session_id 或 timestamp 或 event_type 为空的行
events_clean = events.dropna(subset=['event_id', 'session_id', 'timestamp', 'event_type'])
# 将 qty 的空值填充为 0
events_clean['qty'] = events_clean['qty'].fillna(0)
# 可选：将 product_id 的空值填充为 0（或保留）
events_clean['product_id'] = events_clean['product_id'].fillna(0)
print("重复行数:", events_clean.duplicated().sum())
# 如果有重复，删除
events_clean = events_clean.drop_duplicates()
events_clean['timestamp'] = pd.to_datetime(events_clean['timestamp'])


event_id             0
session_id           0
timestamp            0
event_type           0
product_id       78489
qty             617832
cart_size       716049
payment         727378
discount_pct    727378
amount_usd      727378
dtype: int64
重复行数: 0


### 数据基本信息

In [153]:
print(events_clean['session_id'].describe())

count    760958.000000
mean      60153.502012
std       34613.117088
min           1.000000
25%       30314.250000
50%       60176.000000
75%       90110.000000
max      120000.000000
Name: session_id, dtype: float64


In [154]:
print("清洗后数据形状:", events_clean.shape)
print("各事件类型计数:\n", events_clean['event_type'].value_counts())

清洗后数据形状: (760958, 10)
各事件类型计数:
 event_type
page_view      539343
add_to_cart    143126
checkout        44909
purchase        33580
Name: count, dtype: int64


###  漏斗分析

In [155]:
# 按 session_id 分组，收集每个会话中出现的事件类型集合
session_events = events_clean.groupby('session_id')['event_type'].apply(set)

stages = ['page_view', 'add_to_cart', 'checkout', 'purchase']
funnel_df = pd.DataFrame(index=session_events.index)

for stage in stages:
    funnel_df[stage] = session_events.apply(lambda x: 1 if stage in x else 0)

In [156]:
stage_counts = funnel_df.sum()
print("各阶段会话数：")
print(stage_counts)

各阶段会话数：
page_view      120000
add_to_cart     81518
checkout        44909
purchase        33580
dtype: int64


### 转化率分析

In [157]:
print("\n相对转化率：")
for i in range(1, len(stages)):
    prev = stages[i-1]
    curr = stages[i]
    rate = stage_counts[curr] / stage_counts[prev] if stage_counts[prev] > 0 else 0
    print(f"{prev} → {curr}: {rate:.2%}")


相对转化率：
page_view → add_to_cart: 67.93%
add_to_cart → checkout: 55.09%
checkout → purchase: 74.77%


In [158]:
overall = stage_counts['purchase'] / stage_counts['page_view']
print(f"\n整体转化率 (page_view → purchase): {overall:.2%}")


整体转化率 (page_view → purchase): 27.98%


In [159]:
# 标记加购流失会话（有 add_to_cart 但无 checkout）
abandoned = funnel_df[(funnel_df['add_to_cart'] == 1) & (funnel_df['checkout'] == 0)].index
print(f"加购流失会话数: {len(abandoned)}")
print(f"整体加购流失率: {len(abandoned) / funnel_df['add_to_cart'].sum():.2%}\n")

# 关联 sessions 获取设备、来源、国家
session_features = sessions[['session_id', 'device', 'source', 'country']].drop_duplicates()

# 为每个会话标记是否有加购、是否流失
funnel_sub = funnel_df[['add_to_cart', 'checkout']].reset_index().rename(columns={'index': 'session_id'})
merged = funnel_sub.merge(session_features, on='session_id', how='inner')

# 各设备加购流失率（只考虑有加购的会话）
device_abandon = {}
for device in merged['device'].unique():
    mask_add = merged['add_to_cart'] == 1
    mask_device = merged['device'] == device
    total_add = mask_add & mask_device
    abandoned_add = total_add & (merged['checkout'] == 0)
    if total_add.sum() > 0:
        device_abandon[device] = abandoned_add.sum() / total_add.sum()
    else:
        device_abandon[device] = 0
print("各设备加购流失率:", {k: f"{v:.2%}" for k, v in device_abandon.items()})

# 各来源加购流失率
source_abandon = {}
for source in merged['source'].unique():
    mask_add = merged['add_to_cart'] == 1
    mask_source = merged['source'] == source
    total_add = mask_add & mask_source
    abandoned_add = total_add & (merged['checkout'] == 0)
    if total_add.sum() > 0:
        source_abandon[source] = abandoned_add.sum() / total_add.sum()
    else:
        source_abandon[source] = 0
print("各来源加购流失率:", {k: f"{v:.2%}" for k, v in source_abandon.items()})

# 各国加购流失率（样本量≥100，指加购会话数≥100）
country_abandon = {}
country_add_counts = merged[merged['add_to_cart'] == 1]['country'].value_counts()
valid_countries = country_add_counts[country_add_counts >= 100].index
for country in valid_countries:
    mask_add = merged['add_to_cart'] == 1
    mask_country = merged['country'] == country
    total_add = mask_add & mask_country
    abandoned_add = total_add & (merged['checkout'] == 0)
    if total_add.sum() > 0:
        country_abandon[country] = abandoned_add.sum() / total_add.sum()
    else:
        country_abandon[country] = 0
# 按流失率降序显示前10个
sorted_country = sorted(country_abandon.items(), key=lambda x: x[1], reverse=True)
print("部分国家加购流失率（样本加购数≥100）:")
for c, r in sorted_country[:10]:
    print(f"{c}: {r:.2%}")

加购流失会话数: 36609
整体加购流失率: 44.91%

各设备加购流失率: {'mobile': '44.88%', 'desktop': '44.92%', 'tablet': '45.09%'}
各来源加购流失率: {'email': '44.54%', 'organic': '45.12%', 'direct': '44.71%', 'paid': '44.77%', 'referral': '45.13%', 'social': '45.03%'}
部分国家加购流失率（样本加购数≥100）:
AE: 47.47%
IN: 45.77%
CA: 45.64%
ES: 45.41%
SE: 45.34%
NL: 45.20%
DE: 45.15%
FR: 44.98%
US: 44.86%
GB: 44.75%


#### 加购 → 结账环节流失率高达 44.91%，是整体漏斗中最薄弱的环节。
#### 结账 → 购买转化率相对较高（74.77%），说明一旦用户进入结账流程，支付成功率较高。
#### 设备，来源，国家的流失率范围均在四十五上下，说明在不同设备、来源、国家的用户中，转化率差异较小。
#### 加购后流失是平台级通用问题，而非特定设备、渠道或国家的用户行为异常。

In [160]:
# 商品维度漏斗分析（按每个商品product_id，会话级别去重）

# 确保 product_id 有效
events_clean['product_id'] = events_clean['product_id'].fillna(0).astype(int)
events_nonzero = events_clean[events_clean['product_id'] != 0].copy()

# 按 (session_id, product_id) 聚合事件类型集合
grouped = events_nonzero.groupby(['session_id', 'product_id'])['event_type'].apply(set).reset_index()

def has_event(events_set, event_name):
    return 1 if event_name in events_set else 0

grouped['has_page_view'] = grouped['event_type'].apply(lambda x: has_event(x, 'page_view'))
grouped['has_add_to_cart'] = grouped['event_type'].apply(lambda x: has_event(x, 'add_to_cart'))
grouped['has_checkout'] = grouped['event_type'].apply(lambda x: has_event(x, 'checkout'))
grouped['has_purchase'] = grouped['event_type'].apply(lambda x: has_event(x, 'purchase'))

# 按商品汇总各阶段会话数
product_funnel = grouped.groupby('product_id').agg(
    browse_sessions=('has_page_view', 'sum'),
    add_sessions=('has_add_to_cart', 'sum'),
    checkout_sessions=('has_checkout', 'sum'),
    purchase_sessions=('has_purchase', 'sum')
).reset_index()

# 计算加购率（仅用于筛选，不输出）
product_funnel['add_rate'] = product_funnel['add_sessions'] / product_funnel['browse_sessions']
product_funnel['purchase_rate'] = product_funnel['purchase_sessions'] / product_funnel['browse_sessions']
product_funnel = product_funnel.fillna(0)

# 整体基准转化率
overall_add_rate = 81518 / 120000   # 67.93%
overall_purchase_rate = 33580 / 120000  # 27.98%
threshold_purchase = overall_purchase_rate * 0.8  # 22.38%
threshold_add = overall_add_rate * 0.8           # 54.32%

# 筛选需要优化的商品（浏览会话数>=100，且购买率或加购率低于阈值）
need_optimize = product_funnel[
    (product_funnel['browse_sessions'] >= 100) &
    ((product_funnel['purchase_rate'] < threshold_purchase) | (product_funnel['add_rate'] < threshold_add))
].copy()

need_optimize = need_optimize.sort_values('purchase_rate', ascending=True)

print(f"需要重点优化的商品数量: {len(need_optimize)}")
# 输出列：商品ID、各阶段会话数、加购率（格式化为百分比）
output_cols = ['product_id', 'browse_sessions', 'add_sessions', 'checkout_sessions', 'purchase_sessions', 'add_rate']
need_optimize['add_rate'] = need_optimize['add_rate'].apply(lambda x: f"{x:.2%}")
print(need_optimize[output_cols].head(20).to_string(index=False))

# 关联品类信息（如果有 products 表）
if 'products' in globals():
    product_info = products[['product_id', 'category']].drop_duplicates()
    need_optimize = need_optimize.merge(product_info, on='product_id', how='left')
    print("\n带品类的重点优化商品示例（前10个）：")
    print(need_optimize[['product_id', 'category', 'browse_sessions', 'add_sessions', 'checkout_sessions', 'purchase_sessions']].head(10).to_string(index=False))

需要重点优化的商品数量: 1197
 product_id  browse_sessions  add_sessions  checkout_sessions  purchase_sessions add_rate
          1              187             5                  0                  0    2.67%
        801              366            79                  0                  0   21.58%
        800              344            71                  0                  0   20.64%
        799              295            58                  0                  0   19.66%
        798              364            80                  0                  0   21.98%
        797              527           149                  0                  0   28.27%
        796              690           224                  0                  0   32.46%
        802              464           131                  0                  0   28.23%
        795              651           217                  0                  0   33.33%
        793              567           180                  0                  0  

In [168]:
# 高流失商品价格区间分布（替代之前的统计描述）
high_abandon_products = need_optimize['product_id'].unique()
price_info = products[['product_id', 'price_usd']].drop_duplicates()
high_abandon_price = price_info[price_info['product_id'].isin(high_abandon_products)]

# 定义价格区间（可自行调整）
bins = [0, 20, 50, 100, 200, float('inf')]
labels = ['0-20', '20-50', '50-100', '100-200', '200+']
high_abandon_price['price_bin'] = pd.cut(high_abandon_price['price_usd'], bins=bins, labels=labels, right=False)

# 统计每个区间的商品数量
price_dist = high_abandon_price['price_bin'].value_counts().sort_index()
price_dist_percent = (price_dist / len(high_abandon_price) * 100).round(1)

# 输出结果
print("高流失商品价格区间分布：")
print(pd.DataFrame({'商品数量': price_dist, '占比(%)': price_dist_percent}).to_string())

高流失商品价格区间分布：
           商品数量  占比(%)
price_bin             
0-20        146   12.2
20-50       272   22.7
50-100      281   23.5
100-200     289   24.1
200+        209   17.5


In [169]:
# 如果之前没有计算品类流失率，运行此代码
# 获取所有加购事件
add_events = events_clean[events_clean['event_type'] == 'add_to_cart'][['session_id', 'product_id']].copy()
add_events['product_id'] = add_events['product_id'].astype(int)

# 每个会话是否有结账
session_has_checkout = funnel_df['checkout'].reset_index().rename(columns={'index': 'session_id'})
session_has_checkout.columns = ['session_id', 'has_checkout']

add_with_checkout = add_events.merge(session_has_checkout, on='session_id', how='left')
add_with_checkout['has_checkout'] = add_with_checkout['has_checkout'].fillna(0)
add_with_checkout['is_abandoned'] = (add_with_checkout['has_checkout'] == 0).astype(int)

# 关联品类
product_cat = products[['product_id', 'category']].drop_duplicates()
add_with_cat = add_with_checkout.merge(product_cat, on='product_id', how='left')
add_with_cat['category'] = add_with_cat['category'].fillna('未知')

category_stats = add_with_cat.groupby('category').agg(
    total_add=('is_abandoned', 'count'),
    abandoned_add=('is_abandoned', 'sum')
).reset_index()
category_stats['abandon_rate'] = category_stats['abandoned_add'] / category_stats['total_add']
category_stats = category_stats[category_stats['total_add'] >= 50].sort_values('abandon_rate', ascending=False)

print("各品类加购流失率（加购次数≥50）：")
print(category_stats.to_string(index=False))

各品类加购流失率（加购次数≥50）：
      category  total_add  abandoned_add  abandon_rate
   Electronics       8305           3755      0.452137
          Toys      27642          12480      0.451487
         Books      31008          13981      0.450884
       Fashion      18713           8431      0.450542
Home & Kitchen      17713           7915      0.446847
        Beauty      24491          10878      0.444163
        Sports      15254           6727      0.440999


### 流失是普遍性问题，而非特定渠道或设备的问题。

##### 多维度客单价分析

In [ ]:
import pandas as pd

# 读取 orders 表（请确认文件路径）
orders = pd.read_csv('orders.csv')

# 按国家统计平均订单金额（使用 total_usd 字段）
country_avg = orders.groupby('country')['total_usd'].mean().sort_values(ascending=False)
print("各国平均订单金额（美元）：")
print(country_avg.head(10))

# 按设备统计
device_avg = orders.groupby('device')['total_usd'].mean().sort_values(ascending=False)
print("\n各设备平均订单金额：")
print(device_avg)

# 按来源统计
source_avg = orders.groupby('source')['total_usd'].mean().sort_values(ascending=False)
print("\n各来源平均订单金额：")
print(source_avg)

各国平均订单金额（美元）：
country
ZA    145.026541
PL    141.569704
NL    138.117116
ES    137.337766
MX    136.170158
FR    135.889104
AU    134.815643
US    134.151220
GB    134.136095
CA    133.824803
Name: total_usd, dtype: float64

各设备平均订单金额：
device
mobile     135.165972
tablet     132.557200
desktop    132.068203
Name: total_usd, dtype: float64

各来源平均订单金额：
source
referral    137.190363
paid        136.287183
direct      134.106938
organic     133.265229
social      133.260634
email       129.333495
Name: total_usd, dtype: float64


##### 经过对12万次会话和数万笔订单的多维度分析，我们发现：无论是用户来自哪个国家、使用什么设备、通过何种渠道进入，其购买转化率（约28%）、加购流失率（约30%）以及平均订单金额（约130-145美元）均无明显差异。这表明当前平台的用户行为具有高度一致性，运营优化应聚焦于通用流程（如结账体验、购物车挽回策略），而非针对特定人群。”

#### RFM 分层分析

#### 传统RFM 分层分析

In [ ]:
import pandas as pd
import numpy as np

# 读取 orders 表
orders = pd.read_csv('orders.csv')
orders['order_time'] = pd.to_datetime(orders['order_time'])

# 当前日期
current_date = orders['order_time'].max() + pd.Timedelta(days=1)

# 计算 R、F、M
rfm_trad = orders.groupby('customer_id').agg(
    recency=('order_time', lambda x: (current_date - x.max()).days),
    frequency=('order_id', 'count'),
    monetary=('total_usd', 'sum')
).reset_index()

print("传统 RFM 描述统计：")
print(rfm_trad[['recency', 'frequency', 'monetary']].describe())

# 定义分箱函数：返回分类标签和对应的数值分数
def get_tier_and_score(x, reverse=False):
    # 分箱
    try:
        cats = pd.qcut(x, q=3, labels=['低','中','高'] if not reverse else ['高','中','低'], duplicates='drop')
    except:
        bins = np.linspace(x.min(), x.max(), 4)
        cats = pd.cut(x, bins=bins, labels=['低','中','高'] if not reverse else ['高','中','低'], include_lowest=True)
    # 将分类转换为数值分数：高=3, 中=2, 低=1
    score_map = {'高': 3, '中': 2, '低': 1}
    scores = cats.map(score_map).astype(int)   # 确保整数类型
    return cats, scores

# 计算各维度标签和分数
rfm_trad['R_tier'], rfm_trad['R_score'] = get_tier_and_score(rfm_trad['recency'], reverse=True)
rfm_trad['F_tier'], rfm_trad['F_score'] = get_tier_and_score(rfm_trad['frequency'], reverse=False)
rfm_trad['M_tier'], rfm_trad['M_score'] = get_tier_and_score(rfm_trad['monetary'], reverse=False)

# 计算总分
rfm_trad['total_score'] = rfm_trad['R_score'] + rfm_trad['F_score'] + rfm_trad['M_score']

# 定义分层
def trad_segment(score):
    if score >= 8:
        return '高价值用户'
    elif score >= 6:
        return '中等价值用户'
    elif score >= 4:
        return '低价值用户'
    else:
        return '沉睡用户'

rfm_trad['segment'] = rfm_trad['total_score'].apply(trad_segment)

# 汇总
segment_summary_trad = rfm_trad['segment'].value_counts().reset_index()
segment_summary_trad.columns = ['用户层级', '用户数']
segment_summary_trad['占比'] = (segment_summary_trad['用户数'] / segment_summary_trad['用户数'].sum() * 100).round(1)
segment_summary_trad = segment_summary_trad.sort_values('用户数', ascending=False)

print("\n传统 RFM 用户分层汇总：")
print(segment_summary_trad.to_string(index=False))

# 可选：查看各档位人数分布
print("\n传统 RFM 各档位人数分布：")
print(rfm_trad[['R_tier', 'F_tier', 'M_tier']].apply(pd.Series.value_counts))

传统 RFM 描述统计：
            recency     frequency      monetary
count  16268.000000  16268.000000  16268.000000
mean     776.322781      2.064175    276.199746
std      573.038789      1.122568    265.202586
min        1.000000      1.000000      2.800000
25%      286.000000      1.000000     91.547500
50%      661.000000      2.000000    199.775000
75%     1192.000000      3.000000    376.512500
max     2131.000000      9.000000   3026.420000

传统 RFM 用户分层汇总：
  用户层级  用户数   占比
 低价值用户 7432 45.7
中等价值用户 5424 33.3
  沉睡用户 2515 15.5
 高价值用户  897  5.5

传统 RFM 各档位人数分布：
   R_tier  F_tier  M_tier
中    5414    1740    5424
低    5421   14498    5423
高    5433      30    5421


#### 传统rfm用户画像

In [ ]:
# 假设 rfm_trad 已经包含了每个 customer_id 及其分层
# 我们需要获取每个用户的设备偏好、主要来源、国家等

# 1. 从 orders 表中获取每个用户的国家、设备、来源（取众数）
user_profile = orders.groupby('customer_id').agg(
    country=('country', lambda x: x.mode()[0] if not x.mode().empty else None),
    device=('device', lambda x: x.mode()[0] if not x.mode().empty else None),
    source=('source', lambda x: x.mode()[0] if not x.mode().empty else None),
    avg_order_value=('total_usd', 'mean'),
    total_spent=('total_usd', 'sum'),
    order_count=('order_id', 'count'),
    last_order_date=('order_time', 'max')
).reset_index()

# 2. 合并 RFM 分层
user_profile_with_segment = user_profile.merge(
    rfm_trad[['customer_id', 'segment', 'recency', 'frequency', 'monetary']], 
    on='customer_id', how='left'
)

# 3. 按分层统计画像
segment_profile = user_profile_with_segment.groupby('segment').agg(
    用户数=('customer_id', 'count'),
    平均最近购买间隔_天=('recency', 'mean'),
    平均购买次数=('frequency', 'mean'),
    平均总消费_USD=('monetary', 'mean'),
    平均客单价_USD=('avg_order_value', 'mean'),
    偏好国家=('country', lambda x: x.mode()[0] if not x.mode().empty else 'N/A'),
    偏好设备=('device', lambda x: x.mode()[0] if not x.mode().empty else 'N/A'),
    主要来源=('source', lambda x: x.mode()[0] if not x.mode().empty else 'N/A')
).round(2)

print("各分层用户画像：")
print(segment_profile.to_string())

各分层用户画像：
          用户数  平均最近购买间隔_天  平均购买次数  平均总消费_USD  平均客单价_USD 偏好国家     偏好设备     主要来源
segment                                                                       
中等价值用户   5424      378.14    2.61     427.07     183.80   US   mobile   direct
低价值用户    7432      887.98    1.67     191.78     124.00   US  desktop   direct
沉睡用户     2515     1520.50    1.17      59.35      52.54   US   mobile  organic
高价值用户     897      172.40    4.57     671.35     147.27   US   mobile  organic


#### 行为频次rfm 分层分析

In [ ]:
sessions_sub = sessions[['session_id', 'customer_id']].drop_duplicates()

# 左连接，保留所有事件行（如果某个 session_id 在 sessions 中找不到，则 customer_id 为 NaN，但正常情况下都应匹配）
df_events = events_clean.merge(sessions_sub, on='session_id', how='left')

# 删除 customer_id 为空的行（如果没有匹配的用户，则无法分析）
df_events = df_events.dropna(subset=['customer_id'])
df_events['customer_id'] = df_events['customer_id'].astype(int)

print("合并后事件记录数：", len(df_events))
print("唯一用户数：", df_events['customer_id'].nunique())

# 2. 准备时间列
df_events['发生日期'] = df_events['timestamp'].dt.normalize()
all_users = df_events['customer_id'].unique()

# ---------- 计算 R：距离最近一次购买的天数 ----------
purchase_events = df_events[df_events['event_type'] == 'purchase']
if len(purchase_events) == 0:
    print("警告：没有购买事件，无法计算 R。")
    # 可以设置默认值，但这里先退出
    raise ValueError("数据集中无购买事件")
reference_date = purchase_events['发生日期'].max() + pd.Timedelta(days=1)
R = (reference_date - purchase_events.groupby('customer_id')['发生日期'].max()).dt.days
R_all = R.reindex(all_users, fill_value=999).reset_index()
R_all.columns = ['customer_id', '最近购买间隔_天']

# ---------- 计算 F：每个用户的购买次数 ----------
F = purchase_events.groupby('customer_id').size()
F_all = F.reindex(all_users, fill_value=0).reset_index()
F_all.columns = ['customer_id', '购买次数']

# ---------- 计算 M：每个用户所有行为的总次数 ----------
M = df_events.groupby('customer_id').size().reset_index()
M.columns = ['customer_id', '行为次数总和']

# ---------- 合并 RFM ----------
rfm = R_all.merge(F_all, on='customer_id', how='inner').merge(M, on='customer_id', how='inner')
print("RFM 数据前5行：")
print(rfm.head())

# ---------- 分档（高、中、低）----------
# R: 越小越优，分位数标签 ['高','中','低']（间隔短为高）
rfm['R_tier'] = pd.qcut(rfm['最近购买间隔_天'], q=3, labels=['高','中','低'], duplicates='drop')
# F: 越大越优
rfm['F_tier'] = pd.qcut(rfm['购买次数'], q=3, labels=['低','中','高'], duplicates='drop')
# M: 越大越优
rfm['M_tier'] = pd.qcut(rfm['行为次数总和'], q=3, labels=['低','中','高'], duplicates='drop')

print("\n各档位人数分布：")
print(rfm[['R_tier', 'F_tier', 'M_tier']].apply(pd.Series.value_counts))

# ---------- 组合并映射为6个业务层级 ----------
rfm['组合'] = rfm['R_tier'].astype(str) + '-' + rfm['F_tier'].astype(str) + '-' + rfm['M_tier'].astype(str)

segment_map = {
    # 超级活跃忠诚用户 — R高、F高
    '高-高-高': '超级活跃忠诚用户',
    '高-高-低': '超级活跃忠诚用户',
    '高-中-高': '超级活跃忠诚用户',
    '中-高-高': '超级活跃忠诚用户',
    # 高活跃潜力用户 — M高、R中~高、F低
    '高-低-高': '高活跃潜力用户',
    '中-低-高': '高活跃潜力用户',
    # 新客/轻度用户 — R高、F低~中、M低~中
    '高-低-低': '新客/轻度用户',
    '高-低-中': '新客/轻度用户',
    '高-中-低': '新客/轻度用户',
    '高-中-中': '新客/轻度用户',
    # 活跃但购买力较低用户 — R中~高、F低
    '中-低-中': '活跃但购买力较低用户',
    '中-低-低': '活跃但购买力较低用户',
    # 普通用户 — 各项中等
    '高-高-中': '普通用户',
    '中-高-低': '普通用户',
    '中-高-中': '普通用户',
    '中-中-高': '普通用户',
    '中-中-中': '普通用户',
    '中-中-低': '普通用户',
    # 流失用户 — R低（很久没买）
    '低-高-高': '流失用户',
    '低-高-中': '流失用户',
    '低-高-低': '流失用户',
    '低-中-高': '流失用户',
    '低-中-中': '流失用户',
    '低-中-低': '流失用户',
    '低-低-高': '流失用户',
    '低-低-中': '流失用户',
    '低-低-低': '流失用户',
}
rfm['用户分层'] = rfm['组合'].map(segment_map).fillna('其他')

# 汇总层级分布
segment_summary = rfm['用户分层'].value_counts().reset_index()
segment_summary.columns = ['用户层级', '用户数']
segment_summary['占比'] = (segment_summary['用户数'] / segment_summary['用户数'].sum() * 100).round(1)
segment_summary = segment_summary.sort_values('用户数', ascending=False).reset_index(drop=True)
print("\n用户分层汇总：")
print(segment_summary.to_string(index=False))
segment_summary

合并后事件记录数： 760958
唯一用户数： 19945
RFM 数据前5行：
   customer_id  最近购买间隔_天  购买次数  行为次数总和
0        12360      1500     3      59
1        13917       274     1      25
2         1022       621     2      31
3         2882        83     2      83
4         1286       685     1      39

各档位人数分布：
   R_tier  F_tier  M_tier
中    7919    5315    6539
低    5371    9900    6801
高    6655    4730    6605

用户分层汇总：
      用户层级  用户数   占比
      流失用户 5371 26.9
活跃但购买力较低用户 4665 23.4
  超级活跃忠诚用户 3967 19.9
      普通用户 2658 13.3
   新客/轻度用户 2610 13.1
   高活跃潜力用户  674  3.4


,用户层级,用户数,占比
0,流失用户,5371,26.9
1,活跃但购买力较低用户,4665,23.4
2,超级活跃忠诚用户,3967,19.9
3,普通用户,2658,13.3
4,新客/轻度用户,2610,13.1
5,高活跃潜力用户,674,3.4


#### 行为频次rfm 分层用户画像分析

In [ ]:
import pandas as pd

# 假设 rfm 已经包含 customer_id 和用户分层（从前面代码得到）
# 如果没有，请先运行行为 RFM 代码至 segment_summary

# 1. 获取每个用户的设备、来源、国家偏好（从 sessions 表，取众数）
# 首先，确保 sessions 有 customer_id（可能有重复会话）
user_prefs = sessions.groupby('customer_id').agg(
    device=('device', lambda x: x.mode()[0] if not x.mode().empty else None),
    source=('source', lambda x: x.mode()[0] if not x.mode().empty else None),
    country=('country', lambda x: x.mode()[0] if not x.mode().empty else None)
).reset_index()

# 2. 合并到 rfm 表中（注意 rfm 已包含 R/F/M 数值）
rfm_profile = rfm.merge(user_prefs, on='customer_id', how='left')

# 3. 按用户分层分组汇总
segment_profile = rfm_profile.groupby('用户分层').agg(
    用户数=('customer_id', 'count'),
    平均最近购买间隔_天=('最近购买间隔_天', 'mean'),
    平均购买次数=('购买次数', 'mean'),
    平均行为总次数=('行为次数总和', 'mean'),
    偏好设备=('device', lambda x: x.mode()[0] if not x.mode().empty else 'N/A'),
    偏好来源=('source', lambda x: x.mode()[0] if not x.mode().empty else 'N/A'),
    偏好国家=('country', lambda x: x.mode()[0] if not x.mode().empty else 'N/A')
).round(2)

# 4. 按用户数降序排列
segment_profile = segment_profile.sort_values('用户数', ascending=False)

print("行为频次 RFM 分层用户画像：")
segment_profile

行为频次 RFM 分层用户画像：


,用户数,平均最近购买间隔_天,平均购买次数,平均行为总次数,偏好设备,偏好来源,偏好国家
用户分层,,,,,,,
流失用户,5371,1476.17,1.51,36.44,mobile,organic,US
活跃但购买力较低用户,4665,935.98,0.26,23.19,mobile,direct,US
超级活跃忠诚用户,3967,331.52,3.35,59.75,mobile,organic,US
普通用户,2658,602.54,2.46,39.59,mobile,organic,US
新客/轻度用户,2610,245.86,1.52,30.34,mobile,organic,US
高活跃潜力用户,674,667.63,0.66,52.84,mobile,organic,US


In [179]:
import pandas as pd

# ==================== 1. 漏斗整体 ====================
funnel_overall = pd.DataFrame({
    '阶段': ['浏览页面', '加入购物车', '开始结账', '完成购买'],
    '会话数': [120000, 81518, 44909, 33580],
    '转化率(相对上一阶段)': ['', '67.93%', '55.09%', '74.77%'],
    '转化率(相对浏览)': ['100%', '67.93%', '37.42%', '27.98%']
})

# ==================== 2. 流失率_分维度 ====================
device_data = [
    ('设备', 'mobile', '44.88%', None),
    ('设备', 'desktop', '44.92%', None),
    ('设备', 'tablet', '45.09%', None)
]
source_data = [
    ('来源', 'email', '44.54%', None),
    ('来源', 'organic', '45.12%', None),
    ('来源', 'direct', '44.71%', None),
    ('来源', 'paid', '44.77%', None),
    ('来源', 'referral', '45.13%', None),
    ('来源', 'social', '45.03%', None)
]
country_data = [
    ('国家', 'AE', '47.47%', 2368),
    ('国家', 'IN', '45.77%', 6471),
    ('国家', 'CA', '45.64%', 4064),
    ('国家', 'ES', '45.41%', None),
    ('国家', 'SE', '45.34%', None),
    ('国家', 'NL', '45.20%', None),
    ('国家', 'DE', '45.15%', 5681),
    ('国家', 'FR', '44.98%', 5309),
    ('国家', 'US', '44.86%', 14860),
    ('国家', 'GB', '44.75%', 6406)
]
all_rows = device_data + source_data + country_data
abandon_combined = pd.DataFrame(all_rows, columns=['维度类型', '维度值', '加购流失率', '总加购会话数'])

# ==================== 3. 重点优化商品 ====================
# 检查 need_optimize 是否存在，如果存在则直接使用，否则使用空DataFrame
if 'need_optimize' in globals():
    optimize_products = need_optimize[['product_id', 'category', 'browse_sessions', 'add_sessions', 'checkout_sessions', 'purchase_sessions', 'add_rate']].copy()
    # 如果 add_rate 是数值，才格式化为百分比；否则保留原样（已经是百分比字符串）
    if optimize_products['add_rate'].dtype in ['float64', 'int64']:
        optimize_products['add_rate'] = optimize_products['add_rate'].apply(lambda x: f"{x:.2%}")
else:
    optimize_products = pd.DataFrame(columns=['product_id', 'category', 'browse_sessions', 'add_sessions', 'checkout_sessions', 'purchase_sessions', 'add_rate'])
    print("警告: need_optimize 未定义，重点优化商品表为空")

# ==================== 4. 高流失商品价格区间 ====================
price_dist = pd.DataFrame({
    'price_bin': ['0-20', '20-50', '50-100', '100-200', '200+'],
    'product_count': [146, 272, 281, 289, 209],
    'percentage(%)': [12.2, 22.7, 23.5, 24.1, 17.5]
})

# ==================== 5. 品类加购流失率 ====================
category_stats_df = pd.DataFrame({
    'category': ['Electronics', 'Toys', 'Books', 'Fashion', 'Home & Kitchen', 'Beauty', 'Sports'],
    'total_add': [8305, 27642, 31008, 18713, 17713, 24491, 15254],
    'abandoned_add': [3755, 12480, 13981, 8431, 7915, 10878, 6727],
    'abandon_rate': ['45.21%', '45.15%', '45.09%', '45.05%', '44.68%', '44.42%', '44.10%']
})

# ==================== 6. 客单价_汇总 ====================
country_avg_list = [
    ('国家', 'ZA', 145.03), ('国家', 'PL', 141.57), ('国家', 'NL', 138.12),
    ('国家', 'ES', 137.34), ('国家', 'MX', 136.17), ('国家', 'FR', 135.89),
    ('国家', 'AU', 134.82), ('国家', 'US', 134.15), ('国家', 'GB', 134.14), ('国家', 'CA', 133.82)
]
device_avg_list = [('设备', 'mobile', 135.17), ('设备', 'tablet', 132.56), ('设备', 'desktop', 132.07)]
source_avg_list = [
    ('来源', 'referral', 137.19), ('来源', 'paid', 136.29), ('来源', 'direct', 134.11),
    ('来源', 'organic', 133.27), ('来源', 'social', 133.26), ('来源', 'email', 129.33)
]
avg_price_rows = country_avg_list + device_avg_list + source_avg_list
avg_price_combined = pd.DataFrame(avg_price_rows, columns=['维度类型', '维度值', '平均订单金额(USD)'])

# ==================== 7. 传统RFM分层 ====================
trad_seg = pd.DataFrame({
    '用户层级': ['低价值用户', '中等价值用户', '沉睡用户', '高价值用户'],
    '用户数': [7432, 5424, 2515, 897],
    '占比(%)': [45.7, 33.3, 15.5, 5.5]
})

# ==================== 8. 传统RFM用户画像 ====================
trad_profile = pd.DataFrame({
    '用户层级': ['中等价值用户', '低价值用户', '沉睡用户', '高价值用户'],
    '用户数': [5424, 7432, 2515, 897],
    '平均最近购买间隔_天': [378.14, 887.98, 1520.50, 172.40],
    '平均购买次数': [2.61, 1.67, 1.17, 4.57],
    '平均总消费_USD': [427.07, 191.78, 59.35, 671.35],
    '平均客单价_USD': [183.80, 124.00, 52.54, 147.27],
    '偏好国家': ['US', 'US', 'US', 'US'],
    '偏好设备': ['mobile', 'desktop', 'mobile', 'mobile'],
    '主要来源': ['direct', 'direct', 'organic', 'organic']
})

# ==================== 9. 行为RFM分层 ====================
behav_seg = pd.DataFrame({
    '用户层级': ['流失用户', '活跃但购买力较低用户', '超级活跃忠诚用户', '普通用户', '新客/轻度用户', '高活跃潜力用户'],
    '用户数': [5371, 4665, 3967, 2658, 2610, 674],
    '占比(%)': [26.9, 23.4, 19.9, 13.3, 13.1, 3.4]
})

# ==================== 10. 行为RFM用户画像 ====================
behav_profile = pd.DataFrame({
    '用户分层': ['流失用户', '活跃但购买力较低用户', '超级活跃忠诚用户', '普通用户', '新客/轻度用户', '高活跃潜力用户'],
    '用户数': [5371, 4665, 3967, 2658, 2610, 674],
    '平均最近购买间隔_天': [1476.17, 935.98, 331.52, 602.54, 245.86, 667.63],
    '平均购买次数': [1.51, 0.26, 3.35, 2.46, 1.52, 0.66],
    '平均行为总次数': [36.44, 23.19, 59.75, 39.59, 30.34, 52.84],
    '偏好设备': ['mobile', 'mobile', 'mobile', 'mobile', 'mobile', 'mobile'],
    '偏好来源': ['organic', 'direct', 'organic', 'organic', 'organic', 'organic'],
    '偏好国家': ['US', 'US', 'US', 'US', 'US', 'US']
})

# ==================== 写入Excel ====================
with pd.ExcelWriter('analysis_output.xlsx', engine='openpyxl') as writer:
    funnel_overall.to_excel(writer, sheet_name='漏斗整体', index=False)
    abandon_combined.to_excel(writer, sheet_name='流失率_分维度', index=False)
    optimize_products.to_excel(writer, sheet_name='重点优化商品', index=False)
    price_dist.to_excel(writer, sheet_name='高流失商品价格区间', index=False)
    category_stats_df.to_excel(writer, sheet_name='品类加购流失率', index=False)
    avg_price_combined.to_excel(writer, sheet_name='客单价_汇总', index=False)
    trad_seg.to_excel(writer, sheet_name='传统RFM分层', index=False)
    trad_profile.to_excel(writer, sheet_name='传统RFM用户画像', index=False)
    behav_seg.to_excel(writer, sheet_name='行为RFM分层', index=False)
    behav_profile.to_excel(writer, sheet_name='行为RFM用户画像', index=False)

print("analysis_output.xlsx 已生成，共10个Sheet。")

analysis_output.xlsx 已生成，共10个Sheet。
